# 데이터 정제 (data-cleaning)

원본 `in-vehicle-coupon-recommendation.csv`를 읽어 상태를 확인하고, 관찰 단계에서 결정한 정제 작업을 순서대로 적용해 `coupon_cleaning.csv`를 만든다.
분할·파생 변수·인코딩 같은 전처리는 이 노트북에서 하지 않는다(다음 노트북).

각 단계는 **의도 → 관찰 → 결정** 순서로 기록한다. 관찰 결과의 자세한 내용은 `data.md`, 팀 결정 사항은 회의 문서를 참고한다.

| 순서 | 단계 |
|---|---|
| 1 | 데이터 확인 (크기, 컬럼, 고유값, 요약표) |
| 2 | 거리 컬럼 통합 |
| 3 | 컬럼 제거 |
| 4 | 방문 빈도 결측 처리 |
| 5 | 중복 처리 |
| 6 | 논리 모순 확인 |
| 7 | 응답자 프로필 ID 생성 |
| 8 | 정수 변환 |
| 9 | 정제 결과 저장 |


## 1. 데이터 확인

### csv 파일 가져오고 읽기

**의도.** 원본을 있는 그대로 읽어 크기와 컬럼명을 확인한다.

**읽기 방식의 이유.** 모든 값을 문자열로 읽고(`dtype=str`), 빈칸을 NaN으로 바꾸지 않는다(`keep_default_na=False`).
pandas가 알아서 숫자나 결측으로 해석하면 원본에 어떤 값이 있었는지 알 수 없게 되므로, 확인이 끝난 뒤 필요한 컬럼만 직접 변환한다(정수 변환은 정제 마지막 단계).

**관찰.** 12,684행 × 26열. 설명 변수 25개와 타깃 `Y` 1개.


In [1]:
import pandas as pd

# 불러올 CSV 파일 경로
DATA_FILE = "in-vehicle-coupon-recommendation.csv"

# CSV 파일 읽기
# - dtype=str: 모든 값을 문자열로 읽어 원래 값을 그대로 보존
# - keep_default_na=False: 빈칸을 NaN으로 바꾸지 않고 빈 문자열("")로 유지
# - encoding="utf-8-sig": 파일 앞에 BOM이 있어도 첫 열 이름이 깨지지 않게 처리
df = pd.read_csv(DATA_FILE, dtype=str, keep_default_na=False, encoding="utf-8-sig")

# 데이터 크기 확인 (행 수, 열 수)
print("[행·열 수]", df.shape)

# 열 이름 목록 확인
print("컬럼명")
for col in df.columns:
    print(col)

# 앞부분 5행 미리보기
# print("\n[앞부분 5행]")
# print(df.head())

[행·열 수] (12684, 26)
컬럼명
destination
passanger
weather
temperature
time
coupon
expiration
gender
age
maritalStatus
has_children
education
occupation
income
car
Bar
CoffeeHouse
CarryAway
RestaurantLessThan20
Restaurant20To50
toCoupon_GEQ5min
toCoupon_GEQ15min
toCoupon_GEQ25min
direction_same
direction_opp
Y


### 고유값 .txt 파일 만들기

**의도.** 모든 컬럼의 값 종류, 개수, 비율을 한 번에 보고 이상한 값이나 희소 범주가 있는지 확인한다.

**출력을 파일로 보내는 이유.** 26개 컬럼의 고유값을 모두 출력하면 노트북이 길어지므로 같은 폴더의 `고유값.txt`에 저장한다. 1% 미만 범주는 따로 표시한다.

**관찰.** 오타나 대소문자 불일치 같은 값 오류는 없다. 빈값은 `car`(99.15%)와 방문 빈도 5개 컬럼(0.8~1.7%)에만 있다.
`occupation`은 25개 범주 중 3개가 1% 미만이다.


In [2]:
from pathlib import Path

# CSV 파일과 같은 폴더에 결과 파일 생성
OUTPUT_FILE = Path(DATA_FILE).resolve().parent / "고유값.txt"

with open(OUTPUT_FILE, "w", encoding="utf-8") as f: # 기록을 쌓고싶다면 w -> a로 바꾸면 됨.
    print(f"[데이터] {DATA_FILE}  (행 {df.shape[0]}, 열 {df.shape[1]})", file=f)

    for col in df.columns:
        counts = df[col].value_counts(dropna=False)
        ratios = df[col].value_counts(dropna=False, normalize=True) * 100
        summary = pd.DataFrame({"개수": counts, "비율(%)": ratios.round(2)})

        print(f"\n[{col}] 고유값 {df[col].nunique()}개, 빈 문자열 {(df[col] == '').sum()}개", file=f)
        print(summary.to_string(), file=f)

        rare = summary[summary["비율(%)"] < 1].index.tolist()
        if rare:
            print(f"  -> 1% 미만 희소 범주: {rare}", file=f)

print(f"코드의 출력은 {OUTPUT_FILE} 를 확인하세요.")

코드의 출력은 C:\Users\changhyun\Desktop\UNIV\7학기\LifeData분석과AI\Team-Flow\고유값.txt 를 확인하세요.


### 컬럼 요약

**의도.** 컬럼별 고유값 수, 빈값 수, 최빈값과 그 비율을 표 하나로 모아 정제 대상을 고른다.

**관찰.**
- `toCoupon_GEQ5min`은 고유값이 1개(전부 1)라 정보가 없다.
- `car`는 99.15%가 빈값이다.
- `direction_same`과 `direction_opp`는 고유값 수와 최빈값 비율이 정확히 반대로 같다. 한쪽이 다른 쪽을 결정한다.
- 방문 빈도 5개 컬럼의 최빈값 비율은 27~49%로, 최빈값으로 결측을 채우면 절반 이상 틀린 값이 들어간다.


In [3]:
rows = []
for col in df.columns:
    vc = df[col].value_counts()
    empty = (df[col] == "").sum()
    rows.append({
        "컬럼": col,
        "고유값 수": df[col].nunique(),
        "빈값 수": empty,
        "빈값 비율(%)": round(empty / len(df) * 100, 2),
        "최빈값": vc.index[0],
        "최빈값 비율(%)": round(vc.iloc[0] / len(df) * 100, 1),
    })

summary = pd.DataFrame(rows).set_index("컬럼")
summary

,고유값 수,빈값 수,빈값 비율(%),최빈값,최빈값 비율(%)
컬럼,,,,,
destination,3,0,0.00,No Urgent Place,49.5
passanger,4,0,0.00,Alone,57.6
weather,3,0,0.00,Sunny,79.4
temperature,3,0,0.00,80,51.5
time,5,0,0.00,6PM,25.5
coupon,5,0,0.00,Coffee House,31.5
expiration,2,0,0.00,1d,55.9
gender,2,0,0.00,Female,51.3
age,8,0,0.00,21,20.9


## 2. 데이터 정제

아래 단계는 모두 **데이터를 실제로 바꾸는 작업**이다. 바꾸기 전후의 행 수·열 수를 매 단계 출력해 변화를 추적한다.

원칙은 두 가지다.
1. **원본을 임의로 해석하지 않는다.** 근거 없이 값을 추정해 넣거나, 이상해 보인다는 이유로 지우지 않는다.
2. **처리 순서를 지킨다.** 문자열 비교를 쓰는 확인 작업은 정수 변환보다 앞에, 프로필 ID 생성은 중복 제거보다 뒤에 둔다.


### 2-1. 거리 컬럼 통합

**의도.** `toCoupon_GEQ5min`, `GEQ15min`, `GEQ25min` 세 컬럼이 담은 "가게까지 걸리는 시간" 정보를 순서형 컬럼 하나로 합친다.

**관찰.** GEQ5min은 전부 1이다. GEQ25min이 1이면 GEQ15min도 항상 1이므로 두 컬럼의 합은 0, 1, 2 세 값만 가진다.

| distance | 뜻 | 행 수 |
|---|---|---|
| 0 | 15분 미만 | 5,562 |
| 1 | 15분 이상 25분 미만 | 5,611 |
| 2 | 25분 이상 | 1,511 |

**결정.** `distance = GEQ15min + GEQ25min`을 GEQ25min 바로 오른쪽에 추가한다. 원본 세 컬럼은 이 단계에서 지우지 않고 다음 단계에서 한꺼번에 제거한다.
논리적으로 불가능한 조합(25분 이상인데 15분 미만)이 있으면 경고를 출력하도록 해 두었고, 실제로는 없었다.


In [4]:
# 거리 컬럼 통합
# - toCoupon_GEQ5min은 전부 1이라 정보가 없고,
#   GEQ15min과 GEQ25min은 "15분 이상", "25분 이상"을 각각 0/1로 표시한다.
# - 25분 이상이면 15분 이상도 1이므로, 두 값을 더하면 0, 1, 2의 순서형 값이 된다.
#     0: 15분 미만 / 1: 15분 이상 25분 미만 / 2: 25분 이상
# - 원본 세 컬럼은 그대로 두고, GEQ25min 바로 오른쪽에 distance 컬럼을 추가한다.

# 현재 모든 값이 문자열(dtype=str)이므로 정수로 변환해서 계산한다.
geq15 = df["toCoupon_GEQ15min"].astype(int)
geq25 = df["toCoupon_GEQ25min"].astype(int)

# 25분 이상인데 15분 미만인 행은 논리적으로 불가능하다. 있으면 알려준다.
if ((geq25 == 1) & (geq15 == 0)).any():
    print("논리적으로 불가능한 조합이 있습니다.")

distance = geq15 + geq25

# distance는 0, 1, 2 외의 값을 가질 수 없다.
assert set(distance.unique()) <= {0, 1, 2}, f"예상 밖의 값: {sorted(distance.unique())}"

# GEQ25min 컬럼 위치를 찾아 그 바로 오른쪽에 삽입한다.
insert_at = df.columns.get_loc("toCoupon_GEQ25min") + 1
df.insert(insert_at, "distance", distance)

# 결과 확인: 원본 세 컬럼과 새 컬럼의 조합별 행 수
print(df.groupby(["toCoupon_GEQ5min", "toCoupon_GEQ15min", "toCoupon_GEQ25min", "distance"]).size())

toCoupon_GEQ5min  toCoupon_GEQ15min  toCoupon_GEQ25min  distance
1                 0                  0                  0           5562
                  1                  0                  1           5611
                                     1                  2           1511
dtype: int64


### 2-2. 컬럼 제거

**의도.** 정보가 없거나 다른 컬럼으로 완전히 복원되는 컬럼을 제거한다.

**관찰과 결정.**

| 컬럼 | 근거 | 결정 |
|---|---|---|
| `car` | 99.15%가 빈값. 값이 있는 108행도 응답자 5명의 자유 입력이라 일반화할 수 없음 | 제거 |
| `direction_opp` | 모든 행에서 `direction_same + direction_opp = 1`. 한 컬럼으로 복원됨 | 제거 |
| `toCoupon_GEQ5min` | 전부 1, 정보 없음 | 제거 |
| `toCoupon_GEQ15min`, `GEQ25min` | `distance`로 통합됨 | 제거 |

27열에서 22열이 된다. 행 수는 바뀌지 않는다.


In [5]:
# 컬럼 제거
# 관찰 단계에서 정보가 없거나 중복으로 확인된 컬럼을 제거한다.
#
# 1. car
#    - 결측률 99.15% (12,684행 중 12,576행이 빈값)
#    - 값이 있는 108행도 응답자 5명의 자유 입력 답변이라 일반화할 정보가 없다.
# 2. direction_opp
#    - direction_same과 정반대 값. 모든 행에서 same + opp = 1 이므로
#      direction_same 하나로 완전히 복원된다.
# 3. toCoupon_GEQ5min, toCoupon_GEQ15min, toCoupon_GEQ25min
#    - 세 컬럼의 정보를 distance(0/1/2) 한 컬럼으로 통합했다.
#    - GEQ5min은 모든 행이 1이라 원래부터 정보가 없었다.

drop_cols = [
    "car",
    "direction_opp",
    "toCoupon_GEQ5min",
    "toCoupon_GEQ15min",
    "toCoupon_GEQ25min",
]

# 제거 전후 비교를 위해 기록
before = df.shape

# errors="ignore": 셀을 다시 실행해도 이미 없는 컬럼 때문에 멈추지 않게 한다.
df = df.drop(columns=drop_cols, errors="ignore")

print(f"제거 대상: {drop_cols}")
print(f"열 수: {before[1]} -> {df.shape[1]}  (행 수 {df.shape[0]}, 변화 없음)")
print()
print("남은 컬럼:")
for col in df.columns:
    print(" ", col)

제거 대상: ['car', 'direction_opp', 'toCoupon_GEQ5min', 'toCoupon_GEQ15min', 'toCoupon_GEQ25min']
열 수: 27 -> 22  (행 수 12684, 변화 없음)

남은 컬럼:
  destination
  passanger
  weather
  temperature
  time
  coupon
  expiration
  gender
  age
  maritalStatus
  has_children
  education
  occupation
  income
  Bar
  CoffeeHouse
  CarryAway
  RestaurantLessThan20
  Restaurant20To50
  distance
  direction_same
  Y


### 2-3. 방문 빈도 결측 처리

**의도.** `Bar`, `CoffeeHouse`, `CarryAway`, `RestaurantLessThan20`, `Restaurant20To50`의 빈값을 처리한다.

**관찰.** 빈값이 있는 행은 605개(4.8%)이고, 이 중 42행은 5개가 모두 비어 있고 563행은 한두 개만 비어 있다.
결측 여부와 수락률 사이에는 차이가 없다(결측 행 55.0% vs 나머지 56.9%).

**검토한 선택지.**
- 행 삭제: 응답자 33명의 답 전체를 잃는다. 결측이 Y와 무관하므로 지울 이유가 없다.
- 최빈값 대체: 컬럼별 최빈값 비율이 27~49%라 절반 이상 틀린 값이 들어간다.
- 같은 응답자의 다른 빈도로 추정: 나머지 4개 값에 일관된 패턴이 없어 근거가 없다.

**결정.** 빈값을 `unknown`이라는 별도 범주로 둔다. 원본에 없는 값을 지어내지 않으면서 행도 잃지 않는 방법이다.
처리는 두 단계로 나눠 각각의 규모를 기록한다.


#### 1단계: 5개가 모두 빈 행

**의도.** 방문 습관 정보가 전혀 없는 응답자부터 처리한다. 이들은 추정할 근거 자체가 없으므로 `unknown`이 가장 정직한 값이다.

**관찰.** 42행(응답자 7명). 처리 후 컬럼별 `unknown` 수가 42개로 모두 같아야 정상이고, 일부만 빈 행의 빈값 584개는 아직 남아 있어야 한다.


In [6]:
# 방문 빈도 결측 처리 (1단계: 5개 모두 빈 행)
# - 대상 컬럼: Bar, CoffeeHouse, CarryAway, RestaurantLessThan20, Restaurant20To50
# - keep_default_na=False로 읽었으므로 결측은 NaN이 아니라 빈 문자열("")이다.
# - 삭제하거나 다른 값으로 채우지 않고 "unknown"이라는 별도 범주로 둔다.
#   원본에 없는 값을 만들어 넣지 않으면서 행도 잃지 않기 위해서다.
# - 결측 여부와 수락률 사이에 차이가 없음을 관찰 단계에서 확인했다(55.0% vs 56.9%).
# - 이 셀에서는 5개 컬럼이 모두 빈 행만 처리한다. 이 응답자들은 방문 습관 정보가
#   전혀 없으므로 추정할 근거가 없다. 일부만 빈 행은 다음 셀에서 따로 다룬다.

freq_cols = ["Bar", "CoffeeHouse", "CarryAway", "RestaurantLessThan20", "Restaurant20To50"]

# 처리 전 현황: 행별로 빈 컬럼이 몇 개인지
n_empty = (df[freq_cols] == "").sum(axis=1)
print("행별 빈 컬럼 수 분포:")
print(n_empty.value_counts().sort_index().rename_axis("빈 컬럼 수").rename("행 수").to_string())
print(f"\n5개 모두 빈 행: {(n_empty == 5).sum()}개, 일부만 빈 행: {((n_empty > 0) & (n_empty < 5)).sum()}개")

# 5개 모두 빈 행만 "unknown"으로 치환
all_empty = n_empty == 5
df.loc[all_empty, freq_cols] = "unknown"

# 처리 후 확인
# - 5개 모두 빈 행만 처리했으므로, 일부만 빈 행의 빈 문자열은 그대로 남아 있다.
# - 남은 빈 문자열 수는 (1개 빈 행 수 × 1) + (2개 빈 행 수 × 2)와 같아야 한다.
remaining = (df[freq_cols] == "").sum().sum()
expected = (n_empty[(n_empty > 0) & (n_empty < 5)]).sum()
print(f"\n처리 후 남은 빈 문자열: {remaining}개 (예상 {expected}개)")
print("\n컬럼별 unknown 수 (모두 같아야 정상):")
print((df[freq_cols] == "unknown").sum().to_string())

행별 빈 컬럼 수 분포:
빈 컬럼 수
0    12079
1      542
2       21
5       42

5개 모두 빈 행: 42개, 일부만 빈 행: 563개

처리 후 남은 빈 문자열: 584개 (예상 584개)

컬럼별 unknown 수 (모두 같아야 정상):
Bar                     42
CoffeeHouse             42
CarryAway               42
RestaurantLessThan20    42
Restaurant20To50        42


#### 2단계: 일부만 빈 행

**의도.** 남은 빈값 584개(563행)도 `unknown`으로 통일한다.

**통일해야 하는 이유.** 같은 컬럼에 `""`와 `unknown`이 섞여 있으면 인코딩할 때 두 범주로 갈라진다.

**인코딩에 미치는 영향.** `unknown`은 never~gt8 순서 어디에도 놓을 수 없으므로, 방문 빈도 5개 컬럼은 순서형이 아니라 **원핫 인코딩**한다(unknown이 여섯 번째 범주). 인코딩 자체는 전처리 노트북에서 한다.

**관찰.** 처리 후 빈값 0개. 컬럼별 범주 수는 6개, `unknown` 비율은 0.84~1.71%.


In [7]:
# 방문 빈도 결측 처리 (2단계: 일부만 빈 행)
# - 1단계에서 5개 모두 빈 행을 unknown으로 바꿨고, 한두 컬럼만 빈 행 563개가 남아 있다.
# - 남은 빈칸도 unknown으로 통일한다. 이유:
#   1) 최빈값 대체: 컬럼별 최빈값 비율이 27~49%라 절반 이상 틀린 값이 들어간다.
#   2) 같은 응답자의 다른 빈도로 추정: 나머지 4개 값에 일관된 패턴이 없어 근거가 없다.
#   3) 빈 컬럼이 그 행의 쿠폰 업종과 일치하는 경우는 21.7%뿐이라 핵심 파생 변수 영향이 작다.
#   → 근거 있는 대체가 불가능하므로 결측을 범주로 보존한다.
# - 같은 컬럼 안에 ""와 "unknown"이 섞여 있으면 인코딩 시 두 범주로 갈라지므로 반드시 통일한다.
# - 인코딩 방침: unknown은 never~gt8 순서 어디에도 놓을 수 없으므로, 방문 빈도 5개 컬럼은
#   순서형이 아니라 원핫 인코딩한다(unknown이 여섯 번째 범주가 된다). 인코딩은 전처리 단계에서 수행한다.

freq_cols = ["Bar", "CoffeeHouse", "CarryAway", "RestaurantLessThan20", "Restaurant20To50"]

# 처리 전 현황
before_empty = (df[freq_cols] == "").sum()
print("처리 전 컬럼별 빈 문자열 수:")
print(before_empty.to_string())
print(f"합계: {before_empty.sum()}개")

# 남은 빈 문자열을 모두 unknown으로 치환
df[freq_cols] = df[freq_cols].replace("", "unknown")

# 처리 후 확인: 이제 빈 문자열은 0개여야 한다
remaining = (df[freq_cols] == "").sum().sum()
assert remaining == 0, f"빈 문자열이 {remaining}개 남아 있음"
print(f"\n처리 후 남은 빈 문자열: {remaining}개")

# 컬럼별 범주 구성 확인: 기존 5단계 + unknown = 6개여야 한다
print("\n컬럼별 범주 수와 unknown 수:")
for col in freq_cols:
    n_cat = df[col].nunique()
    n_unknown = (df[col] == "unknown").sum()
    print(f"  {col}: 범주 {n_cat}개, unknown {n_unknown}개 ({n_unknown / len(df) * 100:.2f}%)")

처리 전 컬럼별 빈 문자열 수:
Bar                      65
CoffeeHouse             175
CarryAway               109
RestaurantLessThan20     88
Restaurant20To50        147
합계: 584개

처리 후 남은 빈 문자열: 0개

컬럼별 범주 수와 unknown 수:
  Bar: 범주 6개, unknown 107개 (0.84%)
  CoffeeHouse: 범주 6개, unknown 217개 (1.71%)
  CarryAway: 범주 6개, unknown 151개 (1.19%)
  RestaurantLessThan20: 범주 6개, unknown 130개 (1.02%)
  Restaurant20To50: 범주 6개, unknown 189개 (1.49%)


### 2-4. 중복 처리

**의도.** 완전히 같은 행과, 설명 변수는 같은데 `Y`만 다른 행을 구분해 처리한다.

**관찰.**
- 완전 중복으로 제거되는 행은 74개다. 22행 응답자 안에서 반복된 쌍은 전부 같은 시나리오(출근길·혼자·맑음·80°F·7AM·Carry out·1일)이고 항상 연속된 행이다. 같은 사람에게 같은 질문이 한 번 더 들어간 것으로 보이지만, 의도된 반복인지 설계 실수인지는 알 수 없다.
- 제거 뒤에도 설명 변수가 같고 `Y`만 다른 쌍이 23쌍(46행) 남는다.

**결정 (11/20 회의).**
- 완전 중복은 **삭제**한다. 같은 행이 두 번 있으면 그 응답에 가중치를 두 배 주는 셈이기 때문이다. 12,684행 → 12,610행.
- `Y`만 다른 쌍은 **둘 다 유지**한다. 어느 쪽이 맞는지 판단할 근거가 없다.

**한계.** `Y`만 다른 23쌍은 어떤 모델도 둘 다 맞힐 수 없으므로 정확도 100%는 애초에 불가능하다.
특성이 같은 다른 사람이 같은 시나리오에 답한 것일 가능성도 있어, 삭제된 74행 중 일부는 실제로는 다른 사람의 답이었을 수 있다.


In [8]:
# 중복 처리
# - 완전 중복(모든 칸이 같은 행): 삭제. 회의(11/20) 결정, 합의 문서 3번.
# - 설명 변수는 같고 Y만 다른 쌍: 둘 다 유지. 회의(11/20) 결정, 합의 문서 4번.
#   → 이 쌍들은 어떤 모델도 둘 다 맞힐 수 없으므로 정확도 100%는 불가능하다는 한계를 남긴다.
# - 같은 사람이 같은 시나리오에 두 번 답한 것인지, 특성이 같은 다른 사람인지는 구분할 수 없다.

feature_cols = [c for c in df.columns if c != "Y"]

n_before = len(df)
n_full_dup = df.duplicated().sum()
print(f"완전 중복으로 제거될 행: {n_full_dup}개")

df = df.drop_duplicates().reset_index(drop=True)
print(f"제거 후 행 수: {n_before} → {len(df)}")

# 제거 후에도 남는 'Y만 다른' 쌍 (유지, 한계로 기록)
n_conflict = df.duplicated(subset=feature_cols, keep=False).sum()
print(f"설명 변수는 같고 Y만 다른 행: {n_conflict}개 ({n_conflict // 2}쌍)")

assert df.duplicated().sum() == 0

완전 중복으로 제거될 행: 74개
제거 후 행 수: 12684 → 12610
설명 변수는 같고 Y만 다른 행: 46개 (23쌍)


### 2-5. 논리 모순 확인

**의도.** 설문 응답이라 앞뒤가 맞지 않는 조합이 있을 수 있다. 종류별로 세어 처리 방침을 정한다.

**순서.** 중복 처리 뒤에 센다. 기록하는 숫자가 모델에 실제로 들어가는 행 기준이어야 하기 때문이다(중복 제거 전에 세면 학생·무직 고소득이 1행씩 더 많게 나온다).

**주의.** 값이 아직 문자열이므로 `"0"`, `"Kid(s)"`처럼 비교한다. 이 셀은 정수 변환 셀보다 앞에 있어야 한다.

**관찰.**

| 조건 | 행 수 |
|---|---|
| `passanger=Kid(s)` 이고 `has_children=0` | 17 |
| `occupation=Student` 이고 `income=$100000 or More` | 196 |
| `occupation=Unemployed` 이고 `income=$100000 or More` | 174 |
| `occupation=Retired` 이고 `age` ∈ {26, 36} | 44 |
| `age=below21` 이고 `education=Bachelors degree` | 22 |

하나 이상 해당하는 행은 453개(3.6%)다.

**결정 (11/20 회의).** 모두 유지한다.
- 아이 동승 + 자녀 없음: 본인 자녀가 아닌 아이를 태운 것으로 이해할 수 있다.
- 학생·무직 + 고소득: 설문의 소득은 가구 소득으로 보는 것이 맞다.
- 은퇴 26·36세, 21세 미만 학사: 양이 매우 적고, 데이터를 임의로 해석하지 않기 위해 둔다(결측을 `unknown`으로 둔 것과 같은 원칙).


In [9]:
# 논리 모순 확인
# - 설문 응답이라 앞뒤가 맞지 않는 조합이 있을 수 있다. 종류별로 세어 기록한다.
# - 중복 처리 뒤에 센다. 모델에 실제로 들어가는 행 기준의 숫자를 남기기 위해서다.
# - 주의: 값이 아직 문자열이므로 "0", "Kid(s)"처럼 따옴표로 비교한다. 정수 변환 셀보다 앞에 있어야 한다.
#
# 결정(회의 11/20): 모두 유지. 합의 문서 7번.
# - 아이 동승 + 자녀 없음: 본인 자녀가 아닌 아이를 태운 것으로 이해
# - Student / Unemployed + 고소득: 가구 소득으로 이해
# - Retired 26·36세, below21 학사: 양이 매우 적고, 데이터를 임의로 해석하지 않기 위해 유지
#   (결측을 unknown으로 둔 것과 같은 원칙)

checks = {
    "passanger=Kid(s) & has_children=0": (df["passanger"] == "Kid(s)") & (df["has_children"] == "0"),
    "occupation=Student & income=$100000 or More": (df["occupation"] == "Student") & (df["income"] == "$100000 or More"),
    "occupation=Unemployed & income=$100000 or More": (df["occupation"] == "Unemployed") & (df["income"] == "$100000 or More"),
    "occupation=Retired & age in {26, 36}": (df["occupation"] == "Retired") & df["age"].isin(["26", "36"]),
    "age=below21 & education=Bachelors degree": (df["age"] == "below21") & (df["education"] == "Bachelors degree"),
}

rows = []
for name, mask in checks.items():
    rows.append({"조건": name, "행 수": int(mask.sum()), "비율(%)": round(mask.mean() * 100, 2)})
contradictions = pd.DataFrame(rows).set_index("조건")
print(contradictions.to_string())

any_mask = pd.concat(checks.values(), axis=1).any(axis=1)
print(f"\n하나 이상 해당하는 행: {any_mask.sum()}개 ({any_mask.mean() * 100:.2f}%)")
print("→ 삭제하지 않고 모두 유지한다.")


                                                행 수  비율(%)
조건                                                        
passanger=Kid(s) & has_children=0                17   0.13
occupation=Student & income=$100000 or More     196   1.55
occupation=Unemployed & income=$100000 or More  174   1.38
occupation=Retired & age in {26, 36}             44   0.35
age=below21 & education=Bachelors degree         22   0.17

하나 이상 해당하는 행: 453개 (3.59%)
→ 삭제하지 않고 모두 유지한다.


### 2-6. 응답자 프로필 ID 생성

**의도.** 응답자 ID가 없으므로, 한 사람 안에서 변하지 않는 특성 12개 컬럼의 조합으로 응답자를 구분하는 `profile_id`를 만든다.

**왜 필요한가.** 한 사람이 약 22행을 답했으므로, 행을 무작위로 나누면 같은 사람이 학습과 평가 양쪽에 들어가 성능이 부풀려진다. 이 ID로 **사람 단위 분할**을 한다.
이 ID는 분할에만 쓰고 모델 입력에는 넣지 않는다.

**왜 여기서 만드나.** 중복 제거 뒤에 만들어야 행 수와 ID가 맞는다. 이 작업은 정제가 아니라 평가 설계를 위한 준비이지만, 정제된 행 기준으로 만들어야 해서 이 노트북 끝에 둔다.

**관찰.** 프로필 567개. 정확히 22행인 프로필은 중복 제거 전 489개에서 제거 후 454개가 된다(중복이 빠진 응답자는 21행).
특성이 완전히 같은 다른 사람은 구분할 수 없으므로 실제 응답자는 567명보다 많다(추정 약 589명).


In [10]:
# 응답자 프로필 ID 생성
# - 응답자 ID가 없으므로, 한 사람 안에서 변하지 않는 특성 12칸의 조합으로 응답자를 구분한다.
# - 특성이 완전히 같은 다른 사람은 구분할 수 없다(하한 567명, 추정 589명).
# - 이 ID는 전처리 단계의 그룹 분할(합의 문서 16번)에만 쓰고, 모델 입력에는 넣지 않는다.
# - 중복 제거 뒤에 만들어야 행 수와 ID가 맞는다.

profile_cols = ["gender", "age", "maritalStatus", "has_children", "education", "occupation", "income",
                "Bar", "CoffeeHouse", "CarryAway", "RestaurantLessThan20", "Restaurant20To50"]

df["profile_id"] = pd.factorize(df[profile_cols].agg("|".join, axis=1))[0]

sizes = df["profile_id"].value_counts()
print(f"프로필 수: {df['profile_id'].nunique()}개")
print(f"프로필당 행 수: 중앙값 {sizes.median():.0f}, 최소 {sizes.min()}, 최대 {sizes.max()}")
print(f"정확히 22행인 프로필: {(sizes == 22).sum()}개")


프로필 수: 567개
프로필당 행 수: 중앙값 22, 최소 6, 최대 94
정확히 22행인 프로필: 454개


### 2-7. 정수 변환

**의도.** 첫 셀에서 모든 값을 문자열로 읽었으므로, 숫자여야 할 컬럼을 정수로 바꾼다.

**대상.** `temperature`, `has_children`, `direction_same`, `Y`. `distance`는 만들 때부터 정수다.

**왜 정제 마지막인가.** 문자열 비교를 쓰는 확인 작업(논리 모순)이 모두 끝난 뒤에 바꿔야 한다. 단순 형 변환은 다른 행의 통계를 쓰지 않으므로 분할 전에 해도 정보 누수가 없다.

**결정.** 네 컬럼 모두 정수로 바꾸고, 가능한 값의 집합을 `assert`로 확인한다. 이 셀 아래에서는 이 컬럼들을 `"0"`이 아니라 `0`으로 비교해야 한다.


In [11]:
# 정수 변환
# - 첫 셀에서 모든 칸을 문자열로 읽었으므로, 숫자여야 할 칸을 정수로 바꾼다(합의 문서 9번).
# - 단순 형 변환은 다른 행의 통계를 쓰지 않으므로 분할 전에 해도 정보 누수가 없다.
# - 주의: 이 셀 아래에서는 이 칸들을 "0"이 아니라 0으로 비교해야 한다.
#   문자열 비교를 쓰는 셀(논리 모순 확인)은 이 셀보다 앞에 있다.

int_cols = ["temperature", "has_children", "direction_same", "Y"]
df[int_cols] = df[int_cols].astype(int)

assert set(df["Y"].unique()) <= {0, 1}
assert set(df["temperature"].unique()) <= {30, 55, 80}
assert set(df["has_children"].unique()) <= {0, 1}
assert set(df["direction_same"].unique()) <= {0, 1}
assert set(df["distance"].unique()) <= {0, 1, 2}

print(df.dtypes.to_string())
print(f"\n수락률: {df['Y'].mean() * 100:.1f}%  (기준선 정확도 = {max(df['Y'].mean(), 1 - df['Y'].mean()) * 100:.1f}%)")


destination               str
passanger                 str
weather                   str
temperature             int64
time                      str
coupon                    str
expiration                str
gender                    str
age                       str
maritalStatus             str
has_children            int64
education                 str
occupation                str
income                    str
Bar                       str
CoffeeHouse               str
CarryAway                 str
RestaurantLessThan20      str
Restaurant20To50          str
distance                int64
direction_same          int64
Y                       int64
profile_id              int64

수락률: 56.8%  (기준선 정확도 = 56.8%)


### 2-8. 정제 결과 저장

**의도.** 정제가 끝난 데이터를 `coupon_cleaning.csv`로 저장한다. 전처리 노트북은 이 파일에서 시작하고, 원본 CSV는 건드리지 않는다.

**결과.** 12,610행 × 23열(설명 변수 21개 + `Y` + `profile_id`).

**참고.** `.gitignore`의 `*.csv` 규칙으로 이 파일은 커밋되지 않는다. 팀원은 이 노트북을 전체 실행해 같은 파일을 재현한다.


In [12]:
# 정제 결과 저장
# - 전처리 노트북은 이 파일에서 시작한다. 원본 CSV는 건드리지 않는다.
# - .gitignore의 *.csv 규칙으로 커밋되지 않으므로, 팀원은 이 노트북을 전체 실행해 재현한다.

CLEAN_FILE = Path(DATA_FILE).resolve().parent / "coupon_cleaning.csv"
df.to_csv(CLEAN_FILE, index=False, encoding="utf-8-sig")
print(f"저장: {CLEAN_FILE}  ({df.shape[0]}행 × {df.shape[1]}열)")
print("\n최종 컬럼:")
for col in df.columns:
    print(" ", col, df[col].dtype)


저장: C:\Users\changhyun\Desktop\UNIV\7학기\LifeData분석과AI\Team-Flow\coupon_cleaning.csv  (12610행 × 23열)

최종 컬럼:
  destination str
  passanger str
  weather str
  temperature int64
  time str
  coupon str
  expiration str
  gender str
  age str
  maritalStatus str
  has_children int64
  education str
  occupation str
  income str
  Bar str
  CoffeeHouse str
  CarryAway str
  RestaurantLessThan20 str
  Restaurant20To50 str
  distance int64
  direction_same int64
  Y int64
  profile_id int64
